# Quantum Circuit Simulator: Sprint 2 & 3 Demo


This notebook demonstrates the key features implemented during Sprint 2 and Sprint 3 of our quantum circuit simulator project.

In [1]:
import numpy as np
import sys
sys.path.append('src')

from QuantumCircuit import QuantumCircuit
from gates.registry import GateRegistry
from error_channels.ChannelRegistry import ChannelRegistry
from cbit import CBit

np.set_printoptions(precision=4, suppress=True)

## 1. Gate Registry System (Sprint 2 - Task 31, 33)

We implemented a comprehensive gate registry system that manages all quantum gates in one centralized location.

In [2]:
# Initialize gate registry with default gates
gate_reg = GateRegistry()

print("Available gates in registry:")
print(gate_reg.list())
print("\nGate details:")
print(f"Hadamard gate:\n{gate_reg.get('h')}")

Available gates in registry:
['ccx', 'cx', 'h', 's', 'x', 'y', 'z']

Gate details:
Hadamard gate:
h (1q) Gate:
[[ 0.7071+0.j  0.7071+0.j]
 [ 0.7071+0.j -0.7071+0.j]]


## 2. Multiple Quantum Gates (Sprint 2 - Tasks 6.1-6.5)

Implemented various quantum gates:
- **Pauli Gates** (X, Y, Z)
- **Hadamard Gate** (H)
- **Phase Gate** (S)
- **CNOT Gate** (Controlled-NOT)
- **Toffoli Gate** (CCX)

In [3]:
# Create a simple Bell state circuit: |Φ+⟩ = (|00⟩ + |11⟩)/√2
qc_bell = QuantumCircuit(num_qubits=2)

print("Initial state |00⟩:")
print(qc_bell.get_state())

# Apply Hadamard to qubit 0
qc_bell.add_gate(gate_reg.get('h'), targets=0)
qc_bell.execute()
print("\nAfter Hadamard on qubit 0:")
print(qc_bell.get_state())

# Reset and rebuild for CNOT
qc_bell.reset_state_only()
qc_bell.ops = []  # Clear operations
qc_bell.add_gate(gate_reg.get('h'), targets=0)
# CNOT is a 2-qubit gate: targets=[control, target]
qc_bell.add_gate(gate_reg.get('cx'), targets=[0, 1])
qc_bell.execute()

print("\nBell state |Φ+⟩ = (|00⟩ + |11⟩)/√2:")
print(qc_bell.get_state())
print("\nProbabilities:")
print(qc_bell.measure_probabilities())

Initial state |00⟩:
[1.+0.j 0.+0.j 0.+0.j 0.+0.j]


AttributeError: 'Gate' object has no attribute 'noise'

## 3. Classical Bit Register (Sprint 3 - Task 27)

Implemented a classical bit system to store measurement outcomes.

In [4]:
# Create quantum circuit with classical bits
qc_measure = QuantumCircuit(num_qubits=2, num_cbits=2)

# Prepare |11⟩ state
qc_measure.add_gate(gate_reg.get('x'), targets=0)
qc_measure.add_gate(gate_reg.get('x'), targets=1)
qc_measure.execute()

print("State before measurement:")
print(qc_measure.get_state())
print("\nClassical bits before measurement:")
qc_measure.cbits.print_bits()

AttributeError: 'Gate' object has no attribute 'noise'

## 4. Qubit-Level Measurements (Sprint 3 - Tasks 12.1, 12.2)

Implemented measurements that:
- Collapse the quantum state
- Store results in classical bits
- Support probabilistic outcomes

In [6]:
# Measure both qubits
qc_measure.measure(qubit=0, cbit=0)
qc_measure.measure(qubit=1, cbit=1)
qc_measure.execute()

print("State after measurement:")
print(qc_measure.get_state())
print("\nClassical bits after measurement:")
qc_measure.cbits.print_bits()

# Demonstrate probabilistic measurement on superposition
print("\n" + "="*50)
print("Measuring superposition state:")
qc_super = QuantumCircuit(num_qubits=1, num_cbits=1)
qc_super.add_gate(gate_reg.get('h'), targets=0)  # Create |+⟩ = (|0⟩ + |1⟩)/√2
qc_super.measure(qubit=0, cbit=0)
qc_super.execute()

print(f"Measurement result: {qc_super.cbits.get_bit(0)}")
print(f"Collapsed state: {qc_super.get_state()}")

State after measurement:
[0.+0.j 0.+0.j 0.+0.j 1.+0.j]

Classical bits after measurement:
Classical register 0 contains value: 1
Classical register 1 contains value: 1

Measuring superposition state:
Measurement result: 1
Collapsed state: [0.+0.j 1.+0.j]


## 5. Qubit Reset After Measurement (Sprint 3 - Task 13)

Implemented the ability to reset qubits to |0⟩ after measurement.

In [ ]:
qc_reset = QuantumCircuit(num_qubits=1, num_cbits=1)
qc_reset.add_gate(gate_reg.get('x'), targets=0)
qc_reset.execute()
print("State before measurement: |1⟩")
print(qc_reset.get_state())

qc_reset.measure(qubit=0, cbit=0)
qc_reset.execute()
measurement = qc_reset.cbits.get_bit(0)
print(f"\nMeasurement outcome: {measurement}")
print(f"State after measurement: {qc_reset.get_state()}")

# Reset to |0⟩
qc_reset.reset_qubit(qubit=0)
qc_reset.execute()
print(f"\nState after reset to |0⟩: {qc_reset.get_state()}")

State before measurement: |1⟩
[0.+0.j 1.+0.j]

Measurement outcome: 1
State after measurement: [0.+0.j 1.+0.j]
Measurement outcome was 1, applying X gate to reset qubit 0 to |0>.

State after reset to |0⟩: [1.+0.j 0.+0.j]


## 6. Error Channels (Sprint 2 - Tasks 8.1, 8.2, 8.4, 9, 32, 34)

Implemented quantum noise channels:
- **Bit Flip Channel**: Flips |0⟩ ↔ |1⟩ with probability p
- **Phase Flip Channel**: Applies phase flip with probability p
- **Depolarizing Channel**: General noise model
- **Custom Kraus Operators**: User-defined error channels

In [11]:
# Initialize channel registry
chan_reg = ChannelRegistry()

print("Available channels:")
print(chan_reg.list())

# Demonstrate Bit Flip Channel
print("\n" + "="*60)
print("BIT FLIP CHANNEL")
print("="*60)
bit_flip_30 = chan_reg.get_param('bit_flip').instantiate(0.3)
print(f"{bit_flip_30}")
print(f"Number of Kraus operators: {len(bit_flip_30.kraus_ops)}")

# Apply bit flip to |0⟩ state multiple times to see stochastic behavior
print("\nApplying bit flip (p=0.3) to |0⟩ state:")
psi_0 = np.array([1.0, 0.0], dtype=complex)
for i in range(5):
    result = bit_flip_30.apply_statevector(psi_0.copy())
    print(f"Trial {i+1}: {result} -> measured as |{np.argmax(np.abs(result))}⟩")

# Demonstrate Phase Flip Channel  
print("\n" + "="*60)
print("PHASE FLIP CHANNEL (Bit-Phase Flip)")
print("="*60)
phase_flip_20 = chan_reg.get_param('bit_phase_flip').instantiate(0.2)
print(f"{phase_flip_20}")

# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying phase flip (p=0.2) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(3):
    result = phase_flip_20.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

# Demonstrate Depolarizing Channel
print("\n" + "="*60)
print("DEPOLARIZING CHANNEL")
print("="*60)

depol_15 = chan_reg.get_param('depolarizing').instantiate(0.15)
print(f"{depol_15}")
# Apply to |+⟩ state = (|0⟩ + |1⟩)/√2
print("\nApplying depolarization (p=0.15) to |+⟩ state:")
psi_plus = np.array([1.0, 1.0], dtype=complex) / np.sqrt(2)
print(f"Initial |+⟩ state: {psi_plus}")
for i in range(10):
    result = depol_15.apply_statevector(psi_plus.copy())
    print(f"Trial {i+1}: {result}")

Available channels:
['bit_flip', 'bit_phase_flip', 'depolarizing', 'phase_flip']

BIT FLIP CHANNEL
bit_flip (1q) Channel with 2 Kraus ops
Number of Kraus operators: 2

Applying bit flip (p=0.3) to |0⟩ state:
Trial 1: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 2: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 3: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 4: [1.+0.j 0.+0.j] -> measured as |0⟩
Trial 5: [1.+0.j 0.+0.j] -> measured as |0⟩

PHASE FLIP CHANNEL (Bit-Phase Flip)
bit_phase_flip (1q) Channel with 2 Kraus ops

Applying phase flip (p=0.2) to |+⟩ state:
Initial |+⟩ state: [0.7071+0.j 0.7071+0.j]
Trial 1: [0.7071+0.j 0.7071+0.j]
Trial 2: [0.7071+0.j 0.7071+0.j]
Trial 3: [0.7071+0.j 0.7071+0.j]

DEPOLARIZING CHANNEL
depolarizing (1q) Channel with 4 Kraus ops

Applying depolarization (p=0.15) to |+⟩ state:
Initial |+⟩ state: [0.7071+0.j 0.7071+0.j]
Trial 1: [0.7071+0.j 0.7071+0.j]
Trial 2: [0.-0.7071j 0.+0.7071j]
Trial 3: [0.7071+0.j 0.7071+0.j]
Trial 4: [0.7071+0.j 0.7071+0.j]
Trial 5: [0.7071+

## 7. Noise Integration with Gates (Sprint 3 - Task 30)

Gates can now have noise channels attached, enabling realistic quantum simulations.

In [9]:
from gates.registry import Gate

# Create a noisy Hadamard gate (10% bit flip error)
bit_flip_10 = chan_reg.get_param('bit_flip').instantiate(0.1)
noisy_h = Gate('noisy_h', gate_reg.get('h').matrix, noise=bit_flip_10)

print("Comparing clean vs noisy Hadamard:")
print("\nClean Hadamard on |0⟩:")
psi_clean = np.array([1.0, 0.0], dtype=complex)
result_clean = gate_reg.get('h').apply(psi_clean)
print(f"Result: {result_clean}")
print(f"Probabilities: {np.abs(result_clean)**2}")

print("\nNoisy Hadamard on |0⟩ (with 10% bit flip):")
# Run multiple times to see stochastic behavior
results = []
for i in range(5):
    psi_noisy = np.array([1.0, 0.0], dtype=complex)
    result = noisy_h.apply(psi_noisy)
    results.append(result)
    print(f"Trial {i+1}: {result}")

Comparing clean vs noisy Hadamard:

Clean Hadamard on |0⟩:
Result: [0.7071+0.j 0.7071+0.j]
Probabilities: [0.5 0.5]

Noisy Hadamard on |0⟩ (with 10% bit flip):
Trial 1: [0.7071+0.j 0.7071+0.j]
Trial 2: [0.7071+0.j 0.7071+0.j]
Trial 3: [0.7071+0.j 0.7071+0.j]
Trial 4: [0.7071+0.j 0.7071+0.j]
Trial 5: [0.7071+0.j 0.7071+0.j]


## 8. Simulator Metrics Tracking (Sprint 3 - Multiple tasks)

Comprehensive performance metrics system for analyzing simulator behavior.

In [10]:
# Create circuit with metrics enabled
qc_metrics = QuantumCircuit(num_qubits=3, num_cbits=3, enable_metrics=True)

# Build a GHZ state circuit: (|000⟩ + |111⟩)/√2
qc_metrics.add_gate(gate_reg.get('h'), targets=0)
# CNOT gates are 2-qubit: targets=[control, target]
qc_metrics.add_gate(gate_reg.get('cx'), targets=[0, 1])
qc_metrics.add_gate(gate_reg.get('cx'), targets=[0, 2])
qc_metrics.measure(0, 0)
qc_metrics.measure(1, 1)
qc_metrics.measure(2, 2)

# Execute and display metrics
qc_metrics.execute()
qc_metrics.print_metrics()

print("\nFinal GHZ state (collapsed):")
print(qc_metrics.get_state())
print(f"Measurement outcomes: {[qc_metrics.cbits.get_bit(i) for i in range(3)]}")


QUANTUM SIMULATOR METRICS

[EXECUTION]
  Total execution time: 0.000490 seconds

[MEMORY]
  Initial: 89.42 MB
  Peak:    89.42 MB
  Final:   89.42 MB
  Delta:   +0.00 MB

[OPERATIONS]
  Gates:        3
  Measurements: 3
  Channels:     0
  Total:        6

[TIMING BREAKDOWN]
  Gates:        0.000143s (29.2%)
  Measurements: 0.000232s (47.5%)
  Channels:     0.000000s (0.0%)


Final GHZ state (collapsed):
[0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 0.+0.j 1.+0.j]
Measurement outcomes: [np.int64(1), np.int64(1), np.int64(1)]
